In [1]:
import numpy as np
import os
import xarray as xr
import glob
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import BoundaryNorm
import matplotlib.colors as mcolors
import dask.array as da
import pickle
from scipy.stats import t
import matplotlib.ticker as ticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import seaborn as sns

In [2]:
with open('/home/annierosen16/master.pkl', 'rb') as f:
    
    master = pickle.load(f)

In [3]:
with open('/home/annierosen16/pressure.pkl', 'rb') as f:
    
    pressure = pickle.load(f)
    
with open('/home/annierosen16/sgp_latitudes.pkl', 'rb') as f:
    
    latitude = pickle.load(f)
    
with open('/home/annierosen16/sgp_longitudes.pkl', 'rb') as f:
    
    longitude = pickle.load(f)

In [4]:
# reading in ERA5 u, v, w and q

base_path = '/data/rong4/Data/ERA5/3hourly/quvw_US'

years = [str(year) for year in range(2001, 2019)]

def get_files(folder, component):

    files = glob.glob(os.path.join(base_path, folder, f"era5.{component}.*.nc"))

    filtered_files = [f for f in files if any(year in f for year in years)]
    
    return filtered_files

# Get the files for each component

u_files = get_files('u_component_of_wind', 'u_component_of_wind')

v_files = get_files('v_component_of_wind', 'v_component_of_wind')

w_files = get_files('vertical_velocity', 'vertical_velocity')

q_files = get_files('specific_humidity', 'specific_humidity')

all_files = u_files + v_files + q_files + w_files

# open all datasets at once

convergences = xr.open_mfdataset(all_files, combine='by_coords', chunks={'time': 24})

In [6]:
# filtering for correct domain and time

convvars = convergences.sel(
    
    latitude=slice(40, 29),
    
    longitude=slice(-106, -94)
)

convvars['time'] = convvars['time'] - pd.Timedelta(hours=6)

sub = convvars.where(
    
    (convvars.time.dt.hour.isin([0, 6, 12])) &
    
    (convvars.time.dt.month.isin([5, 6, 7, 8, 9])) &
    
    (convvars.time.dt.year.isin(range(2001, 2019))),
    
    drop=True
)

# Resample to daily frequency and compute the mean

daily_avg = sub.resample(time='D').mean().dropna(dim='time', how='all')

daily_avg['time'].shape

(2754,)

In [ ]:
## reading in surface pressure 

sfc_pressure_path = '/data/rong4/Data/ERA5/3hourly/quvw_US/surface_pressure'

all_files = glob.glob(os.path.join(sfc_pressure_path, "era5.surface_pressure.*.nc"))

# Filter files for years 2001-2018 and months 5-9

filtered_files = [
    
    file for file in all_files 
    
    if 2001 <= int(os.path.basename(file).split(".")[2][:4]) <= 2018  
    
    and 5 <= int(os.path.basename(file).split(".")[2][4:]) <= 9       
]

all_sfc_p = xr.open_mfdataset(filtered_files, combine='by_coords')

all_sfc_p = all_sfc_p.sel(latitude=slice(39.0, 30.0), longitude=slice(-105.0, -95.0))

sfcp = all_sfc_p.assign_coords(time=all_sfc_p.time - pd.Timedelta(hours=6))

sfcp = sfcp.sel(time=sfcp.time[sfcp.time.dt.hour == 6])

print(sfcp['time'].shape)

# average surface pressure per gridpoint

spdf = sfcp['sp'].to_dataframe().reset_index()

spdf_mean = spdf.groupby(['latitude', 'longitude'])['sp'].mean().reset_index(name='average_sp')

In [ ]:
def calculate_horizontal_moisture_convergence(u, v, q, lat, lon):

    R = 6371000  # radius of Earth (meters)
    
    rho_water = 1000 # kg/m^3
    
    seconds_to_day = 60 * 60 * 24 # to convert from s --> day

    latitude_range = np.sin(np.radians(lat[-1])) - np.sin(np.radians(lat[0]))
    
    longitude_range = np.radians(lon[-1]) - np.radians(lon[0])
    
    lon_grid, lat_grid = np.meshgrid(np.radians(lon), np.radians(lat))
    
    delta_y = np.radians(lat[1]) - np.radians(lat[0])
    delta_x = np.radians(lon[1]) - np.radians(lon[0])
    
    # shaped as (#lons, #lats) so first row represents all values for first latitude (eg: 30,30,30...)
    cos_correction = np.cos(lat_grid)
    
    v_scaled = v*cos_correction
    
    dudx = np.zeros_like(u)
    dvdy = np.zeros_like(v)
    
    # Loop through each pressure level
    for k in range(u.shape[0]):
        
        # Central points
        dudx[k, :, 1:-1] = (u[k, :, 2:] - u[k, :, :-2]) / (2 * delta_x)
        dvdy[k, 1:-1, :] = (v_scaled[k, 2:, :] - v_scaled[k, :-2, :]) / (2 * delta_y)
        
        # Left and right boundaries (x-direction)
        dudx[k, :, 0] = (u[k, :, 1] - u[k, :, 0]) / (delta_x)
        dudx[k, :, -1] = (u[k, :, -2] - u[k, :, -1]) / (delta_x)

        # Top and bottom boundaries (y-direction)
        dvdy[k, 0, :] = (v_scaled[k, 1, :] - v_scaled[k, 0, :]) / (delta_y)
        dvdy[k, -1, :] = (v_scaled[k, -2, :] - v_scaled[k, -1, :]) / (delta_y)
        
        dx = R * cos_correction * delta_x
        dy = R * delta_y

        # for every grid point, multiply by the area extent of that grid box 
        dudx[k] = (1/(R*cos_correction)) * dudx[k] * dx * dy
        
        dvdy[k] = (1/(R*cos_correction)) * dvdy[k] * dx * dy
        
    zonal_convergence = -(q * dudx)
    
    meridional_convergence = -(q * dvdy)
    
    total_area = []
    
    for i in dx: 
        
        total_area.append(i*dy)
        
    A = np.sum(total_area)
    
    # Area weighted average
    zonal_convergence_area_weighted = np.sum(zonal_convergence, axis=(1,2))*(1/A)*seconds_to_day
    
    meridional_convergence_area_weighted = np.sum(meridional_convergence, axis=(1,2))*(1/A)*seconds_to_day
    
    # Units = mm/day 
    # Shapes returned: (#pressure levels) because advection is averaged over the domain for every level 

    return zonal_convergence_area_weighted, meridional_convergence_area_weighted

In [7]:
# array = (time = 12 LST, levels, latitude, longitude)

# reversing latitude to be increasing, pressure should go top to bottom 

u = daily_avg['u'].load().data[:, :, ::-1, :]

print("u loaded")

v = daily_avg['v'].load().data[:, :, ::-1, :]

print("v loaded")

q = daily_avg['q'].load().data[:, :, ::-1, :] 

print("q loaded")

w = daily_avg['w'].load().data[:, :, ::-1, :] 

print("w loaded")

latitude = daily_avg['latitude'].data[::-1]

longitude = daily_avg['longitude'].data

pressure = daily_avg['level'].data * 100 # pressure needs to be Pa for convergence calculation

u loaded
v loaded
q loaded
w loaded
